# Notebook 08: Universal Framework for Planetary Polygon Selection

**Central result**: The observed polygon wavenumber $N$ is determined by the intersection of three constraints, all derived from the logarithmic Green's function:

$$N_{\text{selected}} = \max\{N : N \leq N_{\text{Thomson}} \;\text{AND}\; \sin(\pi/N) \geq r_{\text{excl}}/R \;\text{AND}\; n \;\text{is Rossby-stationary}\}$$

Each planet activates a different subset of constraints.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.getcwd()), 'src'))

from fractions import Fraction
from planetary_polygons.core.universal_selection import (
    spectral_gap, spectral_gap_table, packing_bound,
    thomson_bound, kappa_crit, universal_selection,
    saturn_selection, jupiter_north_selection, jupiter_south_selection,
    cross_planetary_table, havelock_eigenvalue,
)
from planetary_polygons.extensions.packing_analysis import (
    exclusion_radius_sensitivity, jupiter_south_packing_analysis,
    jupiter_north_packing_analysis,
)
print("Imports OK")

## 1. Spectral Gap Table (N = 3..12)

The Havelock eigenvalue $\lambda_m = (N-1) - m(N-m)/2$ with critical mode $m = \lfloor N/2 \rfloor$.

In [ ]:
print(f"{'N':>3}  {'m_crit':>6}  {'lambda_min':>12}  {'Status':>10}")
print("-" * 40)
for row in spectral_gap_table(3, 12):
    status = 'STABLE' if row['stable'] else ('marginal' if row['marginal'] else 'UNSTABLE')
    lam = row['lambda_min']
    print(f"{row['N']:>3}  {row['m_crit']:>6}  {str(lam):>12}  {status:>10}")

## 2. Constraint Intersection Table (the key result)

For each planetary system, compute all three bounds and identify the binding constraint.

In [ ]:
table = cross_planetary_table()

print(f"{'System':>20}  {'N_obs':>5}  {'N_sel':>5}  {'N_Thom':>6}  {'N_Ross':>6}  {'N_Pack':>6}  {'Binding':>12}  {'gap':>6}  {'OK':>4}")
print("-" * 85)
for row in table:
    nr = str(row['N_rossby']) if row['N_rossby'] is not None else '  --'
    np_ = str(row['N_packing']) if row['N_packing'] is not None else '  --'
    check = 'YES' if row['match'] else 'NO'
    print(f"{row['system']:>20}  {row['N_observed']:>5}  {row['N_selected']:>5}  "
          f"{row['N_thomson']:>6}  {nr:>6}  {np_:>6}  {row['binding']:>12}  "
          f"{str(row['spectral_gap']):>6}  {check:>4}")

## 3. Packing Sensitivity Analysis for Jupiter South

Invert the packing bound to find the implied exclusion radius, and compare with the cyclone radius + beta-drift.

In [ ]:
result = jupiter_south_packing_analysis()

print("=== Jupiter South Packing Analysis ===")
print(f"  N_observed       = {result['N_observed']}")
print(f"  R_ring           = {result['R_ring_m']:.1e} m")
print(f"  r_cyclone        = {result['r_cyclone_m']:.1e} m")
print(f"  r_excl implied   = {result['r_excl_implied']:.2e} m")
print(f"  r_excl/r_cyclone = {result['ratio_implied']:.2f}")
print(f"  excess (%)       = {result['excess_pct']:.0f}%  (= beta-drift buffer zone)")
print(f"  N_pack (with drift) = {result['N_pack_with_drift']}")
print(f"  N_pack (bare)       = {result['N_pack_bare']}")
print(f"  Consistent: {result['consistent']}")
print()

# Sensitivity: how N_pack changes with r_excl
import math
R = result['R_ring_m']
r_cyc = result['r_cyclone_m']
print(f"{'r_excl/r_cyc':>14}  {'r_excl (km)':>12}  {'N_pack':>6}")
print("-" * 38)
for factor in [1.0, 1.1, 1.2, 1.3, 1.4, 1.5]:
    r_excl = r_cyc * factor
    N_p = packing_bound(R, r_excl)
    print(f"{factor:>14.2f}  {r_excl/1e3:>12.0f}  {N_p:>6}")

## 4. Predictions for Neptune and Uranus

Uranus has a confirmed north polar cyclone (VLA microwave observations, Akins+ 2023). Neptune has transient dark spots but no observed polar vortex crystal. What does the framework predict?

In [ ]:
# Predictions based on Thomson bound (packing/Rossby parameters uncertain for ice giants)
#
# Uranus: north polar cyclone confirmed (Akins+ 2023, VLA microwave).
#   Single cyclone observed so far — no ring/crystal yet.
#   If a vortex crystal forms, Thomson gives the stability ceiling.
#
# Neptune: transient dark spots; no observed polar vortex crystal.

print("=== Predictions (Thomson bound only) ===")
print(f"{'kappa_0/kappa':>14}  {'N_Thomson':>9}  {'gap(N)':>8}")
print("-" * 35)
for kr in [0.0, 0.25, 0.5, 0.75, 1.0]:
    N_t = thomson_bound(kr)
    gap = spectral_gap(N_t)
    print(f"{kr:>14.2f}  {N_t:>9}  {str(gap):>8}")

print()
print("Interpretation:")
print("  - Without central vortex (kappa_ratio=0): max stable N = 7")
print("  - With weak center (kappa_ratio=0.5):     max stable N = 8 (Jupiter north)")
print("  - With strong center (kappa_ratio=1.0):    max stable N = 9")
print("  - Packing may further limit N at ice giants if cyclones are large")
print()
print("Uranus: single polar cyclone confirmed (Akins+ 2023).")
print("  If satellite cyclones form a ring around it, expect N <= 7 (no strong central vortex)")
print("  or N <= 8 if the central cyclone has kappa_0/kappa >= 0.5.")
print("  Packing could further limit N depending on cyclone size vs ring radius.")
print()
print("Neptune: no polar vortex crystal observed.")
print("  If one forms, expect N <= 7 (same Thomson ceiling).")

## 5. Verification

All three systems reproduce their observed N from constraint intersection alone.

In [ ]:
sat = saturn_selection()
jn = jupiter_north_selection()
js = jupiter_south_selection()

print("=== Verification ===")
for name, result, N_obs in [("Saturn", sat, 6), ("Jupiter N", jn, 8), ("Jupiter S", js, 5)]:
    ok = "PASS" if result['N_selected'] == N_obs else "FAIL"
    print(f"  {name:12s}: N_selected={result['N_selected']}, N_observed={N_obs}  [{ok}]")
    print(f"    Binding constraint: {result['binding_constraint']}")
    print(f"    Mechanism: {result['mechanism']}")
    print()